In [45]:
# imports
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import joblib

### Extract Data

In [6]:
file_path1 = 'WorldCupMatches.csv'
file_path2 = 'results.csv'

wc_data = pd.read_csv(file_path1)
intl_data = pd.read_csv(file_path2)

intl_data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


### Clean and Merge Datasets

In [7]:
wc_clean = wc_data.rename(columns={
    "Home Team Name": "home_team",
    "Away Team Name": "away_team",
    "Home Team Goals": "home_score",
    "Away Team Goals": "away_score"
})[["home_team", "away_team", "home_score", "away_score"]]

wc_clean['tournament'] = "FIFA World Cup"

In [8]:
intl_clean = intl_data[["home_team", "away_team", 
                        "home_score", "away_score", "tournament", 'date']]

In [9]:
# Combine datasets
combined_data = pd.concat([wc_clean, intl_clean], ignore_index = True)

In [10]:
# Drop duplicates and null values
combined_data = combined_data.drop_duplicates()
combined_data = combined_data.dropna(subset= ['home_team', 'away_team', 'home_score', 'away_score'])

In [11]:
# Fix team names
combined_data['home_team'] = combined_data['home_team'].replace({
    "United States": "USA",
    "IR Iran": "Iran",
    "Korea Republic": "South Korea"
})

In [12]:
# Convert the data column to datetime and sort chronologically
combined_data['date'] = pd.to_datetime(combined_data['date'])
combined_data = combined_data.sort_values('date').reset_index(drop=True)

combined_data.head()                                                                                                              

,home_team,away_team,home_score,away_score,tournament,date
0,Scotland,England,0.0,0.0,Friendly,1872-11-30
1,England,Scotland,4.0,2.0,Friendly,1873-03-08
2,Scotland,England,2.0,1.0,Friendly,1874-03-07
3,England,Scotland,2.0,2.0,Friendly,1875-03-06
4,Scotland,England,3.0,0.0,Friendly,1876-03-04


### Use competitive matches only

In [13]:
# Remove friendly matches
competitive_tournaments = ["FIFA World Cup","UEFA Euro","Copa America","AFC Asian Cup",
 "Africa Cup of Nations","FIFA World Cup qualification","UEFA Euro qualification","Copa America qualification",
    "African Cup of Nations qualification"]

In [14]:
combined_data = combined_data[combined_data['tournament'].isin(competitive_tournaments)
    ].reset_index(drop = True)

In [15]:
print('Total matches', len(combined_data))
print(combined_data['tournament'].value_counts())

Total matches 16523
tournament
FIFA World Cup qualification            8771
UEFA Euro qualification                 2824
African Cup of Nations qualification    2327
FIFA World Cup                          1792
AFC Asian Cup                            421
UEFA Euro                                388
Name: count, dtype: int64


### Create target variable and new features

In [16]:
# Get the result of the matches
def get_result(row):
    if row['home_score']>row['away_score']:
        return 2  # home win
    elif row['home_score'] == row['away_score']:
        return 1   # draw
    else:
        return 0   # away win
combined_data['result'] = combined_data.apply(get_result, axis=1)

print(combined_data['result'].value_counts())

result
2    8369
0    4576
1    3578
Name: count, dtype: int64


#### Win rates

In [17]:
# Find historical home win rate per team
home_win_rate = combined_data.groupby('home_team')['result'].apply(
    lambda x: (x==2).mean()).rename('home_win_rate').reset_index()

In [18]:
# Find historical away win rate per team
away_win_rate = combined_data.groupby('away_team')['result'].apply(
    lambda x: (x==0).mean()).rename('away_win_rate').reset_index()

In [19]:
# Merge home and away win rates into the data
combined_data = combined_data.merge(home_win_rate, on = 'home_team', how = 'left')
combined_data = combined_data.merge(away_win_rate, on = 'away_team', how = 'left')

# Check if the columns are added
combined_data[['home_team', 'away_team', 'home_win_rate', 'away_win_rate', 'result']].head()

,home_team,away_team,home_win_rate,away_win_rate,result
0,France,Mexico,0.666667,0.324138,2
1,Belgium,United States,0.624365,0.308511,0
2,Peru,Romania,0.423077,0.355556,0
3,Brazil,Yugoslavia,0.729958,0.401709,0
4,Argentina,France,0.693069,0.442786,2


#### Goal Difference

In [20]:
combined_data['goal_diff'] = (combined_data['home_score']-
                             combined_data['away_score'])

In [21]:
home_goal_diff = (combined_data.groupby('home_team')['goal_diff'].mean()
                 .rename('home_avg_goal_diff').reset_index())

In [22]:
away_goal_diff = (combined_data.groupby('away_team')['goal_diff'].mean()
                 .rename('away_avg_goal_diff').reset_index())

In [23]:
combined_data = combined_data.merge(home_goal_diff, on = 'home_team', how = 'left')
combined_data = combined_data.merge(away_goal_diff, on = 'away_team', how = 'left')

#### Win rate over last 10 games

In [24]:
# shift(1) prevents the model from seeing the current match result
# which could cause data leakage
def rolling_win_rate(df, team_col, result_val, window=10):
    return (df.groupby(team_col)['result']
            .transform(lambda x:x.shift(1))
            .rolling(window, min_periods=3)
            .apply(lambda y: (y == result_val).mean()))

In [25]:
combined_data['home_recent_form'] = rolling_win_rate(combined_data, 'home_team', 2)
combined_data['away_recent_form'] = rolling_win_rate(combined_data, 'away_team', 2)

In [26]:
# Drop rows where the rolling form can not be calculated
combined_data = combined_data.dropna(subset= ['home_recent_form', 'away_recent_form'])

print('Total matches after feature engineering', len(combined_data))
combined_data[["home_team", "away_team", "home_recent_form", 
               "away_recent_form", "result"]].head()

Total matches after feature engineering 16504


,home_team,away_team,home_recent_form,away_recent_form,result
11,Belgium,Paraguay,0.2,0.3,0
12,Bolivia,Brazil,0.2,0.3,0
13,Uruguay,Romania,0.3,0.3,2
14,Argentina,Chile,0.4,0.3,2
15,Argentina,United States,0.5,0.2,2


#### Head to head win rate

In [27]:
h_to_h = combined_data.groupby(['home_team', 'away_team']).apply(
    lambda x: (x['result']==2).mean()).rename('h_to_h_home_win_rate').reset_index()

/var/folders/lb/t38plhc13xs7frlj7gxz52t80000gn/T/ipykernel_35049/3934836894.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  h_to_h = combined_data.groupby(['home_team', 'away_team']).apply(


In [28]:
combined_data = combined_data.merge(h_to_h, on = ['home_team', 'away_team'], how = 'left')

In [29]:
# fill teams with no head to head with global home win rate
global_home_win_rate = (combined_data['result']==2).mean()
combined_data['h_to_h_win_rate'] = combined_data['h_to_h_home_win_rate'].fillna(global_home_win_rate)

#### Draw tendency

In [30]:
# how often each draws at home
home_draw_rate = combined_data.groupby('home_team')['result'].apply(
    lambda x: (x==1).mean()).rename('home_draw_rate').reset_index()

In [31]:
# how often each team draws as away team
away_draw_rate = combined_data.groupby('away_team')['result'].apply(
    lambda x: (x==1).mean()).rename('away_draw_rate').reset_index()

In [32]:
combined_data = combined_data.merge(home_draw_rate, on = 'home_team', how = 'left')
combined_data = combined_data.merge(away_draw_rate, on = 'away_team', how = 'left')

### Define X and y, Train Test Split

In [33]:
features = ['home_team', 'away_team', 'tournament', 'home_win_rate', 'away_win_rate',
           'home_avg_goal_diff', 'away_avg_goal_diff', 'home_recent_form', 'away_recent_form',
           'h_to_h_home_win_rate', 'home_draw_rate', 'away_draw_rate']
X = combined_data[features]

y = combined_data['result']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state = 1)

print("Training samples: ", len(X_train))
print("Testing samples: ", len(X_valid))

Training samples:  13203
Testing samples:  3301


### Build preprocessor and columnn transformer

In [34]:
# Numerical features
from sklearn.preprocessing import StandardScaler
numerical_features = ['home_win_rate', 'away_win_rate', 'home_avg_goal_diff',
                     'away_avg_goal_diff', 'home_recent_form', 'away_recent_form',
                     'h_to_h_home_win_rate', 'home_draw_rate', 'away_draw_rate']
numerical_transformer = Pipeline(steps = [('imputer', SimpleImputer(strategy = 'mean')),
                                         ('scaler', StandardScaler())])

In [35]:
# Categorical features
categorical_features = ['home_team', 'away_team', 'tournament']
categorical_transformer = Pipeline(steps = [('imputer', SimpleImputer(strategy= 'most_frequent')),
                                             ('onehot', OneHotEncoder(handle_unknown = 'ignore'))])

In [36]:
# Combine numerical and categorical data into one column transformer
preprocessor = ColumnTransformer(transformers = [('num', numerical_transformer,numerical_features),
                                         ('cat', categorical_transformer, categorical_features)])

### Build and train the pipeline

In [37]:
model = XGBClassifier(n_estimators = 500, learning_rate = 0.05, eval_metric = 'mlogloss',
                     max_depth=4, subsample=0.8, colsample_bytree = 0.8, min_child_weight =3, random_state=1)

In [38]:
my_pipeline = Pipeline(steps = [('preprocessor', preprocessor), ('model', model)])

In [39]:
# Compute sample weights to fix class imbalance
sample_weights = compute_sample_weight(class_weight = 'balanced', y=y_train)

In [40]:
# Train the model
my_pipeline.fit(X_train, y_train, model__sample_weight=sample_weights)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Evaluate the model

In [41]:
y_pred = my_pipeline.predict(X_valid)

print('Accuracy: ', accuracy_score(y_valid, y_pred))
print(classification_report(y_valid, y_pred, target_names = ['Away Win', 'Draw', 'Home Win']))

Accuracy:  0.6752499242653741
              precision    recall  f1-score   support

    Away Win       0.65      0.61      0.63       927
        Draw       0.39      0.52      0.44       663
    Home Win       0.86      0.77      0.81      1711

    accuracy                           0.68      3301
   macro avg       0.63      0.63      0.63      3301
weighted avg       0.70      0.68      0.69      3301



In [42]:
joblib.dump(my_pipeline, 'worldcup_model.pkl')
print('Model saved')

Model saved


In [43]:
# Export team stats lookup table for FastAPI to use
team_stats = combined_data.groupby('home_team').agg(
    home_win_rate = ('result', lambda x: (x==2).mean()),
    away_win_rate = ('result', lambda x: (x==0).mean()),
    home_avg_goal_diff = ('goal_diff', 'mean'),
    home_recent_form = ('home_recent_form', 'mean'),
    away_recent_form = ('away_recent_form', 'mean'),
    h_to_h_home_win_rate = ('h_to_h_home_win_rate', 'mean'),
    home_draw_rate = ('home_draw_rate', 'mean'),
    away_draw_rate = ('away_draw_rate', 'mean')).reset_index()

In [44]:
team_stats.to_csv('team_stats.csv', index = False)
print('Team stats exported for ' + str(len(team_stats)) + ' teams')

Team stats exported for 227 teams
